# Fraud Scoring API (FastAPI)

This notebook mirrors `main.py` for local exploration. **Render still runs `uvicorn api.main:app`** using `main.py`.

Run Jupyter with the **fraud-api** folder as the working directory (or ensure `fraud_model.sav` and `fraud_features.pkl` are discoverable).

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException


def get_fraud_api_root() -> Path:
    """Find folder containing fraud_model.sav (works in Jupyter where __file__ may be missing)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "fraud_model.sav").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find fraud_model.sav. Open/run the notebook with working directory set to the fraud-api folder."
    )


BASE_DIR = get_fraud_api_root()
MODEL_PATH = BASE_DIR / "fraud_model.sav"
FEATURES_PATH = BASE_DIR / "fraud_features.pkl"

print("BASE_DIR:", BASE_DIR)

BASE_DIR: C:\Users\RyanC\OneDrive\Desktop\IS 455 ML\Homework Assignments\fraud-api


In [2]:
app = FastAPI(title="Fraud Scoring API")

model = None
feature_columns = None


def load_artifacts() -> None:
    global model, feature_columns
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Missing model file: {MODEL_PATH.name}")
    if not FEATURES_PATH.exists():
        raise FileNotFoundError(f"Missing feature file: {FEATURES_PATH.name}")

    model = joblib.load(MODEL_PATH)
    feature_columns = joblib.load(FEATURES_PATH)


@app.on_event("startup")
def startup_event() -> None:
    load_artifacts()


@app.get("/health")
def health() -> dict:
    return {
        "ok": True,
        "model_loaded": model is not None,
        "features_loaded": feature_columns is not None,
        "model_path": MODEL_PATH.name,
        "features_path": FEATURES_PATH.name,
    }


@app.post("/score")
def score(payload: dict) -> dict:
    if model is None or feature_columns is None:
        raise HTTPException(status_code=500, detail="Model artifacts are not loaded.")

    rows = payload.get("rows")
    if not isinstance(rows, list) or not rows:
        raise HTTPException(status_code=400, detail="Payload must include a non-empty 'rows' list.")

    df = pd.DataFrame(rows)

    missing = [col for col in feature_columns if col not in df.columns]
    for col in missing:
        df[col] = None

    df = df[feature_columns]

    if not hasattr(model, "predict_proba"):
        raise HTTPException(status_code=500, detail="Loaded model does not support predict_proba.")

    scores = model.predict_proba(df)[:, 1]

    scored = pd.DataFrame(rows).copy()
    scored["fraud_risk"] = scores
    scored = scored.sort_values("fraud_risk", ascending=False)

    top_n = int(payload.get("top_n", 50))
    top_n = max(1, min(top_n, len(scored)))

    return {
        "count": len(scored),
        "top_n": top_n,
        "results": scored.head(top_n).to_dict(orient="records"),
    }

C:\Users\RyanC\AppData\Local\Temp\ipykernel_51372\911979741.py:18: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


## Load model without starting the server
Use this to test scoring in the notebook (skips FastAPI `startup`).

In [3]:
load_artifacts()
health()

{'ok': True,
 'model_loaded': True,
 'features_loaded': True,
 'model_path': 'fraud_model.sav',
 'features_path': 'fraud_features.pkl'}

## Optional: run the API locally
Uncomment and run **one** cell below. Stop the kernel when done.

In [4]:
# import nest_asyncio
# import uvicorn
# nest_asyncio.apply()
# uvicorn.run(app, host="127.0.0.1", port=8000)

In [5]:
# Or from a terminal (recommended for production-like tests):
# cd to fraud-api, then: uvicorn api.main:app --reload --port 8000